In [3]:
import pyomo.environ as pyo

# 1. Khởi tạo mô hình
m = pyo.ConcreteModel()

# 2. Định nghĩa các tập hợp (Sets)
m.J = pyo.Set(initialize=['I', 'D', 'AI', 'H'])
m.S = pyo.Set(initialize=['s1', 's2', 's3', 's4'])

# 3. Tham số (Parameters)
m.p = pyo.Param(m.S, initialize={'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05})
m.beta = pyo.Param(m.J, initialize={'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95})

beta_s_data = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10
}
m.beta_s = pyo.Param(m.S, m.J, initialize=beta_s_data)
m.penalty_rate = pyo.Param(initialize=0.1)

# 4. Biến quyết định (Variables)
m.x = pyo.Var(m.J, within=pyo.NonNegativeReals)
m.y = pyo.Var(m.S, m.J, within=pyo.NonNegativeReals)

# 5. Ràng buộc (Constraints)
m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in m.J) <= 65000)

def budget2_rule(m, s):
    return sum(m.y[s, j] for j in m.J) <= 15000
m.budget2 = pyo.Constraint(m.S, rule=budget2_rule)

def ai_human_link_rule(m, s):
    return m.y[s, 'AI'] <= 0.5 * m.x['H']
m.ai_human_link = pyo.Constraint(m.S, rule=ai_human_link_rule)

# 6. Hàm mục tiêu (Objective)
def obj_rule(m):
    gain_stage1 = sum(m.beta[j] * m.x[j] for j in m.J)
    gain_stage2_expected = 0
    for s in m.S:
        benefit_s = sum(m.beta_s[s, j] * m.y[s, j] for j in m.J)
        penalty_s = m.penalty_rate * sum(m.y[s, j] for j in m.J)
        gain_stage2_expected += m.p[s] * (benefit_s - penalty_s)
    return gain_stage1 + gain_stage2_expected

m.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# 7. Giải bài toán bằng HiGHS
solver = pyo.SolverFactory('appsi_highs')
solver.solve(m)

# 8. Báo cáo quyết định giai đoạn 1
print("-" * 60)
print(f"{'KẾT QUẢ PHÂN BỔ NGÂN SÁCH TỐI ƯU GIAI ĐOẠN 1 (NĂM 0)':^60}")
print("-" * 60)
for j in m.J:
    print(f"  - Đầu tư vào {j:15}: {pyo.value(m.x[j]):>10.2f} tỷ VND")
print("-" * 60)
print(f"Tổng ngân sách thực chi GĐ1: {sum(pyo.value(m.x[j]) for j in m.J):>10.2f} tỷ VND")
print(f"Tổng GDP tăng thêm kỳ vọng:  {pyo.value(m.obj):>10.2f} tỷ VND")
print("-" * 60)

------------------------------------------------------------
    KẾT QUẢ PHÂN BỔ NGÂN SÁCH TỐI ƯU GIAI ĐOẠN 1 (NĂM 0)    
------------------------------------------------------------
  - Đầu tư vào I              :       0.00 tỷ VND
  - Đầu tư vào D              :       0.00 tỷ VND
  - Đầu tư vào AI             :   65000.00 tỷ VND
  - Đầu tư vào H              :       0.00 tỷ VND
------------------------------------------------------------
Tổng ngân sách thực chi GĐ1:   65000.00 tỷ VND
Tổng GDP tăng thêm kỳ vọng:    97075.00 tỷ VND
------------------------------------------------------------


In [7]:
import pyomo.environ as pyo

# 1. Khởi tạo mô hình
m = pyo.ConcreteModel()

# 2. Định nghĩa các tập hợp
m.J = pyo.Set(initialize=['I', 'D', 'AI', 'H'])
m.S = pyo.Set(initialize=['s1', 's2', 's3', 's4'])

# 3. Tham số
m.p = pyo.Param(m.S, initialize={'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05})
m.beta = pyo.Param(m.J, initialize={'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95})

beta_s_data = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10
}
m.beta_s = pyo.Param(m.S, m.J, initialize=beta_s_data)
m.penalty_rate = pyo.Param(initialize=0.1)

# 4. Biến quyết định
m.x = pyo.Var(m.J, within=pyo.NonNegativeReals)
m.y = pyo.Var(m.S, m.J, within=pyo.NonNegativeReals)

# 5. Ràng buộc
# (C1) Tổng ngân sách GĐ 1
m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in m.J) <= 65000)

# (C_Sàn) Ràng buộc sàn đầu tư theo thứ tự I > D > H > AI
min_floors = {'I': 17500, 'D': 17500, 'H': 10000, 'AI': 5000}
def min_floor_rule(m, j):
    return m.x[j] >= min_floors[j]
m.min_floor = pyo.Constraint(m.J, rule=min_floor_rule)

# (C2) Ngân sách Giai đoạn 2
def budget2_rule(m, s):
    return sum(m.y[s, j] for j in m.J) <= 15000
m.budget2 = pyo.Constraint(m.S, rule=budget2_rule)

# (C3) Ràng buộc nhân lực số giới hạn đầu tư AI bổ sung
def ai_human_link_rule(m, s):
    return m.y[s, 'AI'] <= 0.5 * m.x['H']
m.ai_human_link = pyo.Constraint(m.S, rule=ai_human_link_rule)

# 6. Hàm mục tiêu
def obj_rule(m):
    gain_stage1 = sum(m.beta[j] * m.x[j] for j in m.J)
    gain_stage2_expected = 0
    for s in m.S:
        benefit_s = sum(m.beta_s[s, j] * m.y[s, j] for j in m.J)
        penalty_s = m.penalty_rate * sum(m.y[s, j] for j in m.J)
        gain_stage2_expected += m.p[s] * (benefit_s - penalty_s)
    return gain_stage1 + gain_stage2_expected

m.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# 7. Giải bài toán
solver = pyo.SolverFactory('appsi_highs')
solver.solve(m)

# 8. Báo cáo
print("-" * 60)
print(f"{'KẾT QUẢ PHÂN BỔ NGÂN SÁCH TỐI ƯU (CÓ RÀNG BUỘC SÀN)':^60}")
print("-" * 60)
for j in m.J:
    print(f"  - Đầu tư vào {j:15}: {pyo.value(m.x[j]):>10.2f} tỷ VND")
print("-" * 60)
print(f"Tổng ngân sách thực chi GĐ1: {sum(pyo.value(m.x[j]) for j in m.J):>10.2f} tỷ VND")
print(f"Tổng GDP tăng thêm kỳ vọng:  {pyo.value(m.obj):>10.2f} tỷ VND")
print("-" * 60)

------------------------------------------------------------
    KẾT QUẢ PHÂN BỔ NGÂN SÁCH TỐI ƯU (CÓ RÀNG BUỘC SÀN)     
------------------------------------------------------------
  - Đầu tư vào I              :   17500.00 tỷ VND
  - Đầu tư vào D              :   17500.00 tỷ VND
  - Đầu tư vào AI             :   20000.00 tỷ VND
  - Đầu tư vào H              :   10000.00 tỷ VND
------------------------------------------------------------
Tổng ngân sách thực chi GĐ1:   65000.00 tỷ VND
Tổng GDP tăng thêm kỳ vọng:    87712.50 tỷ VND
------------------------------------------------------------


In [8]:
import pyomo.environ as pyo
import pandas as pd

# 1. Khởi tạo dữ liệu gốc theo đề bài
scenarios = ['s1', 's2', 's3', 's4']
items = ['I', 'D', 'AI', 'H']
probs = {'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05}

# Hệ số Beta Giai đoạn 1 (Lợi suất cơ sở thực tế)
beta_base = {'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95}

# Hệ số Beta hiệu chỉnh theo kịch bản Giai đoạn 2 (Bảng 10.4)
beta_s_data = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10
}

# CẬP NHẬT: Ràng buộc sàn đầu tư theo đề xuất cải tiến mới của bạn
min_floors = {'I': 17500, 'D': 17500, 'H': 10000, 'AI': 5000}
penalty_rate = 0.1

# 2. Hàm xây dựng và giải mô hình tổng quát
def solve_model(mode='SP', target_scenario=None):
    m = pyo.ConcreteModel()
    m.x = pyo.Var(items, within=pyo.NonNegativeReals)
    
    # Ràng buộc ngân sách Giai đoạn 1 (65.000 tỷ)
    m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in items) <= 65000)
    # Áp dụng mức sàn mới cho Giai đoạn 1
    m.floor = pyo.Constraint(items, rule=lambda m, j: m.x[j] >= min_floors[j])

    # Hàm tính lợi ích Giai đoạn 1
    first_stage_gain = sum(beta_base[j] * m.x[j] for j in items)

    if mode == 'SP':
        # MÔ HÌNH NGẪU NHIÊN LƯỠNG GIAI ĐOẠN (Stochastic Programming)
        m.y = pyo.Var(scenarios, items, within=pyo.NonNegativeReals)
        m.budget2 = pyo.Constraint(scenarios, rule=lambda m, s: sum(m.y[s,j] for j in items) <= 15000)
        m.link = pyo.Constraint(scenarios, rule=lambda m, s: m.y[s,'AI'] <= 0.5 * m.x['H'])
        
        # Kỳ vọng toán học của Giai đoạn 2 trên cả 4 kịch bản
        second_stage_expected_gain = sum(
            probs[s] * (sum(beta_s_data[s,j] * m.y[s,j] for j in items) - penalty_rate * sum(m.y[s,j] for j in items))
            for s in scenarios
        )
        m.obj = pyo.Objective(expr=first_stage_gain + second_stage_expected_gain, sense=pyo.maximize)

    elif mode in ['EV', 'WS']:
        # MÔ HÌNH XÁC ĐỊNH (WS: Từng kịch bản riêng rẽ | EV: Kịch bản trung bình)
        m.y = pyo.Var(items, within=pyo.NonNegativeReals)
        m.budget2 = pyo.Constraint(expr=sum(m.y[j] for j in items) <= 15000)
        m.link = pyo.Constraint(expr=m.y['AI'] <= 0.5 * m.x['H'])
        
        if mode == 'EV':
            # Tính toán ma trận hệ số Beta kỳ vọng (Trung bình trọng số)
            beta_s_target = {j: sum(probs[s] * beta_s_data[s,j] for s in scenarios) for j in items}
        else:
            # Lấy chính xác hệ số Beta của kịch bản mục tiêu (Thông tin hoàn hảo)
            beta_s_target = {j: beta_s_data[target_scenario, j] for j in items}
            
        second_stage_gain = sum(beta_s_target[j] * m.y[j] for j in items) - penalty_rate * sum(m.y[j] for j in items)
        m.obj = pyo.Objective(expr=first_stage_gain + second_stage_gain, sense=pyo.maximize)

    # Giải mô hình bằng solver HiGHS
    pyo.SolverFactory('appsi_highs').solve(m)
    return {j: pyo.value(m.x[j]) for j in items}

# 3. Thực thi tính toán so sánh các mô hình
results = {}

# (b) Giải bài toán xác định với từng kịch bản riêng rẽ (Wait-and-See)
for s in scenarios:
    results[f'Wait-and-See ({s})'] = solve_model(mode='WS', target_scenario=s)

# (a) Lời giải kỳ vọng (EV - Expected Value)
results['Expected Value (EV)'] = solve_model(mode='EV')

# (c) Lời giải Stochastic (SP)
results['Stochastic (SP)'] = solve_model(mode='SP')

# 4. Xuất kết quả trực quan dưới dạng bảng DataFrame
df = pd.DataFrame(results).T
print("=" * 70)
print(f"{'BẢNG SO SÁNH QUYẾT ĐỊNH ĐẦU TƯ GIAI ĐOẠN 1 (TỶ VNĐ)':^70}")
print("=" * 70)
print(df.round(2))
print("=" * 70)

         BẢNG SO SÁNH QUYẾT ĐỊNH ĐẦU TƯ GIAI ĐOẠN 1 (TỶ VNĐ)          
                           I        D       AI        H
Wait-and-See (s1)    17500.0  17500.0  20000.0  10000.0
Wait-and-See (s2)    17500.0  17500.0  20000.0  10000.0
Wait-and-See (s3)    17500.0  17500.0  20000.0  10000.0
Wait-and-See (s4)    17500.0  17500.0  20000.0  10000.0
Expected Value (EV)  17500.0  17500.0  20000.0  10000.0
Stochastic (SP)      17500.0  17500.0  20000.0  10000.0


In [10]:
import pyomo.environ as pyo

# --- 1. DỮ LIỆU ĐẦU VÀO ---
scenarios = ['s1', 's2', 's3', 's4']
items = ['I', 'D', 'AI', 'H']
probs = {'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05}
beta_base = {'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95}
beta_s_data = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10
}
min_floors = {'I': 17500, 'D': 17500, 'H': 10000, 'AI': 5000}
penalty_rate = 0.1

solver = pyo.SolverFactory('appsi_highs')

# --- 2. CÁC HÀM TẠO MÔ HÌNH ---
def create_sp_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(items, within=pyo.NonNegativeReals)
    m.y = pyo.Var(scenarios, items, within=pyo.NonNegativeReals)
    m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in items) <= 65000)
    m.floor = pyo.Constraint(items, rule=lambda m, j: m.x[j] >= min_floors[j])
    m.budget2 = pyo.Constraint(scenarios, rule=lambda m, s: sum(m.y[s,j] for j in items) <= 15000)
    m.link = pyo.Constraint(scenarios, rule=lambda m, s: m.y[s,'AI'] <= 0.5 * m.x['H'])
    
    gain1 = sum(beta_base[j] * m.x[j] for j in items)
    gain2 = sum(probs[s] * (sum(beta_s_data[s,j] * m.y[s,j] for j in items) - penalty_rate * sum(m.y[s,j] for j in items)) for s in scenarios)
    m.obj = pyo.Objective(expr=gain1 + gain2, sense=pyo.maximize)
    return m

def create_ev_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(items, within=pyo.NonNegativeReals)
    m.y = pyo.Var(items, within=pyo.NonNegativeReals)
    m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in items) <= 65000)
    m.floor = pyo.Constraint(items, rule=lambda m, j: m.x[j] >= min_floors[j])
    m.budget2 = pyo.Constraint(expr=sum(m.y[j] for j in items) <= 15000)
    m.link = pyo.Constraint(expr=m.y['AI'] <= 0.5 * m.x['H'])
    
    beta_s_target = {j: sum(probs[s] * beta_s_data[s,j] for s in scenarios) for j in items}
    gain1 = sum(beta_base[j] * m.x[j] for j in items)
    gain2 = sum(beta_s_target[j] * m.y[j] for j in items) - penalty_rate * sum(m.y[j] for j in items)
    m.obj = pyo.Objective(expr=gain1 + gain2, sense=pyo.maximize)
    return m

def create_ws_model(s):
    m = pyo.ConcreteModel()
    m.x = pyo.Var(items, within=pyo.NonNegativeReals)
    m.y = pyo.Var(items, within=pyo.NonNegativeReals)
    m.budget1 = pyo.Constraint(expr=sum(m.x[j] for j in items) <= 65000)
    m.floor = pyo.Constraint(items, rule=lambda m, j: m.x[j] >= min_floors[j])
    m.budget2 = pyo.Constraint(expr=sum(m.y[j] for j in items) <= 15000)
    m.link = pyo.Constraint(expr=m.y['AI'] <= 0.5 * m.x['H'])
    
    gain1 = sum(beta_base[j] * m.x[j] for j in items)
    gain2 = sum(beta_s_data[s,j] * m.y[j] for j in items) - penalty_rate * sum(m.y[j] for j in items)
    m.obj = pyo.Objective(expr=gain1 + gain2, sense=pyo.maximize)
    return m

# --- 3. TÍNH TOÁN CÁC CHỈ SỐ ---
# 3.1. Giải mô hình SP để lấy Giá trị SP
m_sp = create_sp_model()
solver.solve(m_sp)
SP_value = pyo.value(m_sp.obj)

# 3.2. Tính EEV
# Bước 1: Giải mô hình EV để lấy bộ quyết định x_EV
m_ev = create_ev_model()
solver.solve(m_ev)
x_ev_vals = {j: pyo.value(m_ev.x[j]) for j in items}

# Bước 2: Ép mô hình SP phải dùng bộ quyết định x_EV
m_eev = create_sp_model()
for j in items:
    m_eev.x[j].fix(x_ev_vals[j]) # FIX cứng GĐ 1
solver.solve(m_eev)
EEV_value = pyo.value(m_eev.obj)

# 3.3. Tính Lợi ích kỳ vọng của Wait-and-See (WS)
WS_value = 0
for s in scenarios:
    m_ws = create_ws_model(s)
    solver.solve(m_ws)
    WS_value += probs[s] * pyo.value(m_ws.obj)

# 3.4. Tính VSS và EVPI
VSS = SP_value - EEV_value
EVPI = WS_value - SP_value

# --- 4. IN KẾT QUẢ ---
print("-" * 50)
print("CÁC CHỈ SỐ ĐÁNH GIÁ MÔ HÌNH NGẪU NHIÊN")
print("-" * 50)
print(f"Giá trị SP (Stochastic)       : {SP_value:,.2f} tỷ VND")
print(f"Giá trị EEV (EV đưa vào thực tế): {EEV_value:,.2f} tỷ VND")
print(f"Giá trị WS (Wait-and-See)     : {WS_value:,.2f} tỷ VND")
print("-" * 50)
print(f"Chỉ số VSS (SP - EEV)         : {VSS:,.2f} tỷ VND")
print(f"Chỉ số EVPI (WS - SP)         : {EVPI:,.2f} tỷ VND")
print("-" * 50)

--------------------------------------------------
CÁC CHỈ SỐ ĐÁNH GIÁ MÔ HÌNH NGẪU NHIÊN
--------------------------------------------------
Giá trị SP (Stochastic)       : 87,712.50 tỷ VND
Giá trị EEV (EV đưa vào thực tế): 87,712.50 tỷ VND
Giá trị WS (Wait-and-See)     : 87,712.50 tỷ VND
--------------------------------------------------
Chỉ số VSS (SP - EEV)         : 0.00 tỷ VND
Chỉ số EVPI (WS - SP)         : 0.00 tỷ VND
--------------------------------------------------


In [13]:
import pyomo.environ as pyo
import pandas as pd

# -----------------------------------------------------------------
# 1. KHỞI TẠO DỮ LIỆU (Bao gồm cải tiến Ràng buộc sàn của bạn)
# -----------------------------------------------------------------
J = ['I', 'D', 'AI', 'H']
S = ['s1', 's2', 's3', 's4']
probs = {'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05}
beta = {'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95}

beta_s = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10
}

# CẢI TIẾN CỦA BẠN: Ràng buộc sàn đầu tư
min_floors = {'I': 17500, 'D': 17500, 'H': 10000, 'AI': 5000}
penalty_rate = 0.1
solver = pyo.SolverFactory('appsi_highs')

# -----------------------------------------------------------------
# BƯỚC 1: GIẢI MÔ HÌNH SP (Lời giải Ngẫu nhiên - Code của bạn)
# -----------------------------------------------------------------
m_sp = pyo.ConcreteModel()
m_sp.x = pyo.Var(J, within=pyo.NonNegativeReals)
m_sp.y = pyo.Var(S, J, within=pyo.NonNegativeReals)

m_sp.budget1 = pyo.Constraint(expr=sum(m_sp.x[j] for j in J) <= 65000)
m_sp.floor = pyo.Constraint(J, rule=lambda m, j: m.x[j] >= min_floors[j])
m_sp.budget2 = pyo.Constraint(S, rule=lambda m, s: sum(m.y[s, j] for j in J) <= 15000)
m_sp.link = pyo.Constraint(S, rule=lambda m, s: m.y[s, 'AI'] <= 0.5 * m.x['H'])

gain1 = sum(beta[j] * m_sp.x[j] for j in J)
gain2_exp = sum(probs[s] * (sum(beta_s[s, j] * m_sp.y[s, j] for j in J) - penalty_rate * sum(m_sp.y[s, j] for j in J)) for s in S)
m_sp.obj = pyo.Objective(expr=gain1 + gain2_exp, sense=pyo.maximize)
solver.solve(m_sp)
x_sp = {j: pyo.value(m_sp.x[j]) for j in J}

# -----------------------------------------------------------------
# BƯỚC 2: TÍNH Z*(s) - GDP LÝ TƯỞNG CỦA TỪNG KỊCH BẢN (WAIT-AND-SEE)
# -----------------------------------------------------------------
Z_star = {}
for s in S:
    m_ws = pyo.ConcreteModel()
    m_ws.x = pyo.Var(J, within=pyo.NonNegativeReals)
    m_ws.y = pyo.Var(J, within=pyo.NonNegativeReals)
    
    m_ws.budget1 = pyo.Constraint(expr=sum(m_ws.x[j] for j in J) <= 65000)
    m_ws.floor = pyo.Constraint(J, rule=lambda m, j: m.x[j] >= min_floors[j])
    m_ws.budget2 = pyo.Constraint(expr=sum(m_ws.y[j] for j in J) <= 15000)
    m_ws.link = pyo.Constraint(expr=m_ws.y['AI'] <= 0.5 * m_ws.x['H'])
    
    g1 = sum(beta[j] * m_ws.x[j] for j in J)
    g2 = sum(beta_s[s, j] * m_ws.y[j] for j in J) - penalty_rate * sum(m_ws.y[j] for j in J)
    m_ws.obj = pyo.Objective(expr=g1 + g2, sense=pyo.maximize)
    solver.solve(m_ws)
    Z_star[s] = pyo.value(m_ws.obj)

# -----------------------------------------------------------------
# BƯỚC 3: GIẢI MÔ HÌNH ROBUST OPTIMIZATION (Minimax Regret)
# -----------------------------------------------------------------
m_ro = pyo.ConcreteModel()
m_ro.x = pyo.Var(J, within=pyo.NonNegativeReals)
m_ro.y = pyo.Var(S, J, within=pyo.NonNegativeReals)
m_ro.R_max = pyo.Var(within=pyo.Reals) # Biến đại diện cho Nỗi hối tiếc lớn nhất

m_ro.budget1 = pyo.Constraint(expr=sum(m_ro.x[j] for j in J) <= 65000)
m_ro.floor = pyo.Constraint(J, rule=lambda m, j: m.x[j] >= min_floors[j])
m_ro.budget2 = pyo.Constraint(S, rule=lambda m, s: sum(m_ro.y[s, j] for j in J) <= 15000)
m_ro.link = pyo.Constraint(S, rule=lambda m, s: m_ro.y[s, 'AI'] <= 0.5 * m_ro.x['H'])

# Ràng buộc tuyến tính: Regret lớn nhất phải lớn hơn hoặc bằng Regret của TỪNG kịch bản
def regret_rule(m, s):
    actual_gdp = sum(beta[j] * m.x[j] for j in J) + sum(beta_s[s, j] * m.y[s, j] for j in J) - penalty_rate * sum(m.y[s, j] for j in J)
    return m.R_max >= Z_star[s] - actual_gdp
m_ro.regret_constraints = pyo.Constraint(S, rule=regret_rule)

# Hàm mục tiêu: CỰC TIỂU HÓA R_max
m_ro.obj = pyo.Objective(expr=m_ro.R_max, sense=pyo.minimize)
solver.solve(m_ro)
x_ro = {j: pyo.value(m_ro.x[j]) for j in J}
max_regret_value = pyo.value(m_ro.R_max)

# -----------------------------------------------------------------
# 4. IN BÁO CÁO SO SÁNH
# -----------------------------------------------------------------
df = pd.DataFrame({
    'Hạng mục': ['I (Hạ tầng)', 'D (Dữ liệu)', 'AI', 'H (Nhân lực)'],
    'Mức sàn (min_floors)': [min_floors[j] for j in J],
    'Stochastic (SP)': [x_sp[j] for j in J],
    'Robust (RO)': [x_ro[j] for j in J]
})

print("=" * 65)
print(f"{'SO SÁNH QUYẾT ĐỊNH PHÂN BỔ GIAI ĐOẠN 1 (Tỷ VND)':^65}")
print("=" * 65)
print(df.to_string(index=False))
print("-" * 65)
print(f"Nỗi hối tiếc lớn nhất (Worst-case Regret) của RO: {max_regret_value:.2f} tỷ VND")
print("=" * 65)

         SO SÁNH QUYẾT ĐỊNH PHÂN BỔ GIAI ĐOẠN 1 (Tỷ VND)         
    Hạng mục  Mức sàn (min_floors)  Stochastic (SP)  Robust (RO)
 I (Hạ tầng)                 17500          17500.0      17500.0
 D (Dữ liệu)                 17500          17500.0      17500.0
          AI                  5000          20000.0      20000.0
H (Nhân lực)                 10000          10000.0      10000.0
-----------------------------------------------------------------
Nỗi hối tiếc lớn nhất (Worst-case Regret) của RO: -0.00 tỷ VND
